# NB9: Advanced Fugu Orchestration

NB6 teaches the first rung: rule-based heterogeneous routing. Sakana Fugu goes further. It studies learned orchestrator models, dynamic workflow scaffolds, collective intelligence, isolation between workflows, persistent memory, and adaptive topology selection. This notebook bridges that gap with offline teaching adapters: a scaffold generator, a debate moderator, a workflow restructurer, and a simple learner that updates routing rules from history.

## 1. The Limitation of Rule-Based Routing

A keyword router can be useful, but it is not learned orchestration. It can choose a model class, but it does not invent a workflow, aggregate disagreement, or adapt after repeated failures. The production mindset is: routing is one decision inside orchestration, not the whole system.

In [ ]:
from collections import defaultdict
from enum import Enum
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class ModelClass(str, Enum):
    FAST_CHEAP = "fast-cheap"
    BALANCED = "balanced"
    REASONING = "reasoning"
    CODE_SPECIALIST = "code-specialist"

class AgentRole(str, Enum):
    PRODUCT_MANAGER = "product_manager"
    CODER = "coder"
    QA = "qa"
    SECURITY_REVIEWER = "security_reviewer"
    COMPLIANCE_AGENT = "compliance_agent"
    REVIEWER = "reviewer"
    HUMAN = "human"

class TaskComplexity(StrictModel):
    complexity: Literal["simple", "medium", "complex"]
    requires_reasoning: bool
    code_heavy: bool
    risk_level: Literal["low", "medium", "high"]
    word_count: int

class WorkflowScaffold(StrictModel):
    agents_needed: list[AgentRole]
    execution_order: list[tuple[AgentRole, AgentRole]]
    parallel_groups: list[list[AgentRole]] = Field(default_factory=list)
    estimated_total_cost: float
    estimated_total_latency: float

class DebateRecord(StrictModel):
    topic: str
    agent_a_position: str
    agent_b_position: str
    points_of_agreement: list[str]
    points_of_contention: list[str]
    final_decision: str
    escalation_required: bool = False

class EscalationTicket(StrictModel):
    reason: str
    source: Literal["debate", "repair_budget", "human_review"]
    evidence: list[str] = Field(default_factory=list)

class RoutingHistory(StrictModel):
    task_description: str
    task_features: dict[str, float]
    chosen_model: ModelClass
    actual_cost_usd: float
    actual_latency_seconds: float
    success_score: float = Field(ge=0.0, le=1.0)

## 2. Dynamic Scaffold Generation

Fugu-style orchestration asks a larger question than "which model should answer?" It asks, "what workflow should exist for this task?" A simple task may need one analyst. A security audit needs a reviewer and a compliance agent. The scaffold is created from the task, not hard-coded into the application.

In [ ]:
class ScaffoldGenerator:
    def analyze_task(self, task: str) -> TaskComplexity:
        text = task.lower()
        words = text.split()
        high_risk_terms = {"security", "audit", "auth", "compliance", "payment", "pii"}
        code_terms = {"implement", "code", "api", "function", "test"}
        reasoning_terms = {"design", "strategy", "architecture", "tradeoff", "audit"}

        risk_level = "high" if any(term in text for term in high_risk_terms) else "medium"
        if len(words) <= 4 and risk_level != "high":
            complexity = "simple"
        elif len(words) <= 10 and "strategy" not in text and risk_level != "high":
            complexity = "medium"
        else:
            complexity = "complex"

        return TaskComplexity(
            complexity=complexity,
            requires_reasoning=any(term in text for term in reasoning_terms) or risk_level == "high",
            code_heavy=any(term in text for term in code_terms),
            risk_level=risk_level,
            word_count=len(words),
        )

    def generate(self, task: str, complexity: TaskComplexity) -> WorkflowScaffold:
        text = task.lower()

        if "security" in text or "audit" in text or "compliance" in text:
            return WorkflowScaffold(
                agents_needed=[
                    AgentRole.PRODUCT_MANAGER,
                    AgentRole.SECURITY_REVIEWER,
                    AgentRole.COMPLIANCE_AGENT,
                    AgentRole.REVIEWER,
                ],
                execution_order=[
                    (AgentRole.PRODUCT_MANAGER, AgentRole.SECURITY_REVIEWER),
                    (AgentRole.PRODUCT_MANAGER, AgentRole.COMPLIANCE_AGENT),
                    (AgentRole.SECURITY_REVIEWER, AgentRole.REVIEWER),
                    (AgentRole.COMPLIANCE_AGENT, AgentRole.REVIEWER),
                ],
                parallel_groups=[[AgentRole.SECURITY_REVIEWER, AgentRole.COMPLIANCE_AGENT]],
                estimated_total_cost=0.42,
                estimated_total_latency=8.0,
            )

        if complexity.complexity == "simple":
            return WorkflowScaffold(
                agents_needed=[AgentRole.PRODUCT_MANAGER],
                execution_order=[],
                parallel_groups=[],
                estimated_total_cost=0.01,
                estimated_total_latency=0.5,
            )

        if complexity.complexity == "medium":
            return WorkflowScaffold(
                agents_needed=[AgentRole.PRODUCT_MANAGER, AgentRole.CODER],
                execution_order=[(AgentRole.PRODUCT_MANAGER, AgentRole.CODER)],
                parallel_groups=[],
                estimated_total_cost=0.08,
                estimated_total_latency=2.0,
            )

        return WorkflowScaffold(
            agents_needed=[
                AgentRole.PRODUCT_MANAGER,
                AgentRole.CODER,
                AgentRole.QA,
                AgentRole.REVIEWER,
            ],
            execution_order=[
                (AgentRole.PRODUCT_MANAGER, AgentRole.CODER),
                (AgentRole.PRODUCT_MANAGER, AgentRole.QA),
                (AgentRole.CODER, AgentRole.REVIEWER),
                (AgentRole.QA, AgentRole.REVIEWER),
            ],
            parallel_groups=[[AgentRole.CODER, AgentRole.QA]],
            estimated_total_cost=0.22,
            estimated_total_latency=5.0,
        )

generator = ScaffoldGenerator()
tasks = [
    "Classify ticket",
    "Draft release plan",
    "Run security audit for auth workflow",
]

for task in tasks:
    complexity = generator.analyze_task(task)
    scaffold = generator.generate(task, complexity)
    print(f"Task: {task}")
    print(f"Complexity: {complexity.complexity}, risk={complexity.risk_level}")
    print(f"Agents: {[agent.value for agent in scaffold.agents_needed]}")
    print(f"Parallel: {[[agent.value for agent in group] for group in scaffold.parallel_groups]}")
    print()

security_scaffold = generator.generate(
    "security audit",
    generator.analyze_task("security audit"),
)
assert AgentRole.SECURITY_REVIEWER in security_scaffold.agents_needed
assert AgentRole.COMPLIANCE_AGENT in security_scaffold.agents_needed

## 3. Collective Intelligence via Debate

For complex decisions, one agent's answer is often not enough. A debate pattern asks multiple agents for positions, extracts agreement and contention, and either synthesizes a decision or escalates. This is not free-form argument; it is structured disagreement with a typed record.

In [ ]:
class DebateModerator:
    STOPWORDS = {
        "the", "a", "an", "for", "and", "or", "to", "use", "with",
        "data", "system", "because", "when", "is", "are",
    }

    def _keywords(self, text: str) -> set[str]:
        cleaned = "".join(ch.lower() if ch.isalnum() else " " for ch in text)
        return {
            word
            for word in cleaned.split()
            if len(word) > 2 and word not in self.STOPWORDS
        }

    def moderate(self, topic: str, position_a: str, position_b: str) -> DebateRecord:
        keywords_a = self._keywords(position_a)
        keywords_b = self._keywords(position_b)
        agreements = sorted(keywords_a & keywords_b)
        contentions = sorted((keywords_a ^ keywords_b))[:8]

        combined = f"{position_a} {position_b}".lower()
        if "postgresql" in combined and ("mongodb" in combined or "nosql" in combined):
            final = "Use PostgreSQL for relational data and Redis for caching."
            escalation = False
        elif not agreements and len(contentions) > 6:
            final = "Escalate to a human architect; the agents disagree without enough overlap."
            escalation = True
        else:
            final = "Adopt the shared requirements and run a small validation experiment."
            escalation = False

        return DebateRecord(
            topic=topic,
            agent_a_position=position_a,
            agent_b_position=position_b,
            points_of_agreement=agreements,
            points_of_contention=contentions,
            final_decision=final,
            escalation_required=escalation,
        )

moderator = DebateModerator()
debate = moderator.moderate(
    topic="PostgreSQL vs MongoDB for customer platform",
    position_a="Use PostgreSQL because customer orders need relational consistency.",
    position_b="Use MongoDB or NoSQL for flexible profiles, but add Redis for fast caching.",
)
print(debate.model_dump_json(indent=2))
assert debate.final_decision == "Use PostgreSQL for relational data and Redis for caching."

unresolved_debate = moderator.moderate(
    topic="Release readiness",
    position_a="Coder says ship now because tests passed.",
    position_b="Security reviewer says block release because auth token handling is incomplete.",
)
if unresolved_debate.escalation_required:
    ticket = EscalationTicket(
        reason="Debate moderator could not resolve release readiness.",
        source="debate",
        evidence=unresolved_debate.points_of_contention,
    )
    print(ticket.model_dump_json(indent=2))

## 4. Adaptive Topology Selection

Static pipelines hide failure. Adaptive orchestration changes the workflow when evidence says the current topology is wrong. If QA fails because security was missing, add Security and Compliance in parallel before the final reviewer.

In [ ]:
class WorkflowRestructurer:
    def restructure(self, scaffold: WorkflowScaffold, failure_reason: str) -> WorkflowScaffold:
        text = failure_reason.lower()
        agents = list(scaffold.agents_needed)
        edges = list(scaffold.execution_order)
        parallel_groups = [list(group) for group in scaffold.parallel_groups]

        if "security" in text or "compliance" in text:
            for agent in [AgentRole.SECURITY_REVIEWER, AgentRole.COMPLIANCE_AGENT]:
                if agent not in agents:
                    agents.append(agent)
            if AgentRole.REVIEWER not in agents:
                agents.append(AgentRole.REVIEWER)
            edges.extend([
                (AgentRole.SECURITY_REVIEWER, AgentRole.REVIEWER),
                (AgentRole.COMPLIANCE_AGENT, AgentRole.REVIEWER),
            ])
            parallel_groups.append([AgentRole.SECURITY_REVIEWER, AgentRole.COMPLIANCE_AGENT])

        return WorkflowScaffold(
            agents_needed=agents,
            execution_order=edges,
            parallel_groups=parallel_groups,
            estimated_total_cost=round(scaffold.estimated_total_cost + 0.18, 2),
            estimated_total_latency=round(scaffold.estimated_total_latency + 3.0, 2),
        )

initial = generator.generate(
    "Draft API implementation plan",
    generator.analyze_task("Draft API implementation plan"),
)
restructured = WorkflowRestructurer().restructure(
    initial,
    "QA failed because security and compliance review were missing.",
)
print("Before:", [agent.value for agent in initial.agents_needed])
print("After:", [agent.value for agent in restructured.agents_needed])
assert AgentRole.SECURITY_REVIEWER in restructured.agents_needed
assert AgentRole.COMPLIANCE_AGENT in restructured.agents_needed

## 5. From Heuristics to Learning

A production orchestrator should learn from history. This simple learner is intentionally small: it looks at the last ten routing decisions, identifies repeated low-success choices, and updates the routing table. It is not the full Sakana Fugu method, but it shows the bridge from static rules to learned orchestration.

In [ ]:
class SimpleLearner:
    def __init__(self):
        self.learned_rules: dict[str, ModelClass] = {}

    def extract_features(self, task: str) -> dict[str, float]:
        text = task.lower()
        return {
            "has_security": 1.0 if "security" in text or "audit" in text else 0.0,
            "has_strategy": 1.0 if "strategy" in text or "design" in text else 0.0,
            "has_bulk": 1.0 if "500" in text or "variations" in text else 0.0,
            "word_count": float(len(text.split())),
        }

    def route(self, task: str) -> ModelClass:
        features = self.extract_features(task)
        for feature, model in self.learned_rules.items():
            if features.get(feature, 0.0) > 0.0:
                return model
        if features["has_bulk"]:
            return ModelClass.FAST_CHEAP
        if features["has_strategy"]:
            return ModelClass.REASONING
        return ModelClass.FAST_CHEAP

    def analyze_history(self, history: list[RoutingHistory]) -> dict[ModelClass, list[str]]:
        buckets: dict[ModelClass, dict[str, list[float]]] = defaultdict(lambda: defaultdict(list))
        for item in history[-10:]:
            for feature, value in item.task_features.items():
                if value > 0.0 and feature != "word_count":
                    buckets[item.chosen_model][feature].append(item.success_score)

        patterns: dict[ModelClass, list[str]] = {}
        for model, feature_scores in buckets.items():
            strong_features = []
            for feature, scores in feature_scores.items():
                avg_score = sum(scores) / len(scores)
                if avg_score >= 0.8:
                    strong_features.append(feature)
            patterns[model] = strong_features
        return patterns

    def update_rules(self, history: list[RoutingHistory]) -> None:
        recent = history[-10:]
        failure_counts: dict[str, int] = defaultdict(int)

        for item in recent:
            for feature, value in item.task_features.items():
                if value <= 0.0 or feature == "word_count":
                    continue
                if item.success_score < 0.5:
                    failure_counts[feature] += 1

        for feature, count in failure_counts.items():
            if count >= 5 and feature in {"has_security", "has_strategy"}:
                self.learned_rules[feature] = ModelClass.REASONING
            elif count >= 5 and feature == "has_bulk":
                self.learned_rules[feature] = ModelClass.BALANCED

learner = SimpleLearner()
task = "security audit for login workflow"
print("Before learning:", learner.route(task).value)

failed_history = [
    RoutingHistory(
        task_description=f"security audit attempt {i}",
        task_features=learner.extract_features(task),
        chosen_model=ModelClass.FAST_CHEAP,
        actual_cost_usd=0.01,
        actual_latency_seconds=0.5,
        success_score=0.2,
    )
    for i in range(10)
]

learner.update_rules(failed_history)
print("Learned rules:", {k: v.value for k, v in learner.learned_rules.items()})
print("After learning:", learner.route(task).value)
assert learner.route(task) == ModelClass.REASONING

## 6. Intelligent Orchestrator Pattern

The next step beyond deterministic routing is an LLM manager that reads team state and chooses a typed action. The LLM does not execute arbitrary code. It emits a `DelegationDecision`; the framework validates that decision, then calls approved tools such as `delegate_to_coder`, `delegate_to_qa`, or `escalate_to_human`.

In [ ]:
class TaskSpec(StrictModel):
    goal: str
    acceptance_criteria: list[str]
    risk_level: Literal["low", "medium", "high"] = "medium"

class CodePatch(StrictModel):
    files: dict[str, str]
    rationale: str

class TeamStateSnapshot(StrictModel):
    task: TaskSpec
    patch: CodePatch | None = None
    repair_attempts: int = 0
    max_repairs: int = 2
    last_error: str | None = None

class DelegationDecision(StrictModel):
    action: Literal["delegate", "escalate", "review"]
    target_agent: AgentRole
    payload: dict
    rationale: str

def delegate_to_coder(task: TaskSpec) -> CodePatch:
    return CodePatch(
        files={"app.py": "def add(a, b): return a + b"},
        rationale=f"Coder implemented task: {task.goal}",
    )

def delegate_to_qa(patch: CodePatch) -> dict:
    passed = "return a + b" in patch.files.get("app.py", "")
    return {"passed": passed, "log": "QA passed" if passed else "QA failed"}

def escalate_to_human(reason: str) -> dict:
    return {"status": "ESCALATED_TO_HUMAN", "reason": reason}

def mock_llm_orchestrator(state: TeamStateSnapshot) -> dict:
    """Teaching adapter for an LLM manager with tool-calling intent."""
    if state.repair_attempts >= state.max_repairs:
        return {
            "action": "escalate",
            "target_agent": AgentRole.HUMAN,
            "payload": {
                "reason": f"Repair budget exhausted after {state.repair_attempts} attempts."
            },
            "rationale": "The workflow should stop and request human review.",
        }
    if state.patch is None:
        return {
            "action": "delegate",
            "target_agent": AgentRole.CODER,
            "payload": {"task": state.task.model_dump(mode="json")},
            "rationale": "No patch exists yet, so delegate implementation to coder.",
        }
    return {
        "action": "review",
        "target_agent": AgentRole.QA,
        "payload": {"patch": state.patch.model_dump(mode="json")},
        "rationale": "A patch exists and must be tested before release.",
    }

def execute_delegation(decision: DelegationDecision) -> dict:
    if decision.action == "delegate" and decision.target_agent == AgentRole.CODER:
        task = TaskSpec.model_validate(decision.payload["task"])
        return {"patch": delegate_to_coder(task).model_dump(mode="json")}
    if decision.action == "review" and decision.target_agent == AgentRole.QA:
        patch = CodePatch.model_validate(decision.payload["patch"])
        return {"qa": delegate_to_qa(patch)}
    if decision.action == "escalate" and decision.target_agent == AgentRole.HUMAN:
        return escalate_to_human(str(decision.payload["reason"]))
    raise ValueError(f"Unsupported delegation decision: {decision}")

active_state = TeamStateSnapshot(
    task=TaskSpec(
        goal="Implement add endpoint",
        acceptance_criteria=["Return correct sums", "Include tests"],
    )
)
decision = DelegationDecision.model_validate(mock_llm_orchestrator(active_state))
print(decision.model_dump_json(indent=2))
coder_result = execute_delegation(decision)
assert "patch" in coder_result

exhausted_state = TeamStateSnapshot(
    task=active_state.task,
    repair_attempts=2,
    max_repairs=2,
    last_error="QA failed twice",
)
escalation_decision = DelegationDecision.model_validate(
    mock_llm_orchestrator(exhausted_state)
)
escalation_result = execute_delegation(escalation_decision)
print(escalation_result)
assert escalation_decision.action == "escalate"
assert escalation_result["status"] == "ESCALATED_TO_HUMAN"

## 🧪 Exercises: The Internal Brain

**The Story:** Your first agent chain worked when every task was simple. Then the CEO asked for a secure login system, the Coder wanted to ship, the Security Reviewer wanted to block, and the old linear pipeline had no place to resolve the conflict. A real manager brain must decide the workflow shape, organize disagreement, and know when to escalate.

**Your Mission:** Agents must do more than pass notes. They must scaffold workflows, debate conflicting evidence, and route based on complexity.

### Exercise 9.1: The Dynamic Scaffold

**Problem:** A linear chain fails when the task is complex. A simple task may need one agent; a high-risk task may need Product Manager → Coder → Security Reviewer → QA → Reviewer.

**Task:** Implement or extend `WorkflowScaffold`.

**Input:** `TaskComplexity` from NB6-style routing.

**Output:** `agents_needed`, `execution_order`, and `parallel_groups`.

**The Nail:** If `risk_level == "high"`, the scaffold must automatically insert `SecurityReviewer` before `Reviewer`.

### Exercise 9.2: The Debate Protocol

**Problem:** The Coder says "ship"; the Security Reviewer says "block." Who wins?

**Task:** Implement or extend `DebateRecord` and `DebateModerator`.

**Required fields:** `agent_a_position`, `agent_b_position`, `points_of_contention`, `final_decision`, and `escalation_required`.

**The Nail:** If the Moderator cannot resolve the contention, it must emit an `EscalationTicket` for the human steering layer in NB10.

### Bridge to NB10

NB9 is the Internal Brain. It builds the workflow and manages disagreement. NB10 is the External Steering Wheel. When NB9 cannot resolve contention, NB10 lets the human update TeamLog or veto the release.

**The Takeaway:** Advanced orchestration is not "more agents." It is the management layer that decides which agents are needed, how they connect, how disagreement is resolved, and when a human must step in.